# Semantic Correspondence — Colab (H100)

End-to-end: clone, Drive symlinks for `runs/` and `checkpoints/`, SPair-71k, weights, `config.yaml`, full pipeline, analysis.

**Resume:** re-run all cells; stage skip uses `runs/pipeline_state.json`; training uses `checkpoints/*_resume.pt` at `resume_save_interval`.

### GPU

Runtime → Change runtime type → **GPU** (H100 recommended).

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch;\nprint(torch.__version__);\nprint('cuda_available=', torch.cuda.is_available());\nprint('torch_cuda=', torch.version.cuda)",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)

need_reinstall = ("cuda_available= False" in result.stdout) and ("torch_cuda= None" in result.stdout)

if need_reinstall:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--force-reinstall",
            "torch",
            "torchvision",
            "--index-url",
            "https://download.pytorch.org/whl/cu121",
        ],
        check=True,
    )

import torch

print("torch", torch.__version__)
print("cuda_available=", torch.cuda.is_available())
print("torch_cuda=", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu=", torch.cuda.get_device_name(0))
    p = torch.cuda.get_device_properties(0)
    print(f"capability={p.major}.{p.minor}  total_mem_GB={p.total_memory / (1024**3):.1f}")


### Ephemeral cleanup

Removes `/content/Semantic-Correspondence` only (not Drive).

In [ ]:
import shutil
from pathlib import Path

FORCE_CLEAN = True
for p in (Path("/content/Semantic-Correspondence"), Path("/content/sample_data")):
    if FORCE_CLEAN and p.exists():
        print("Removing:", p)
        shutil.rmtree(p, ignore_errors=True)


### Clone repository

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/Semantic-Correspondence")
REPO_URL = "https://github.com/lucaosti/Semantic-Correspondence.git"

if not REPO_DIR.is_dir():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repo exists:", REPO_DIR)

os.chdir(REPO_DIR)
print("REPO =", REPO_DIR.resolve())


### Drive: persist `runs/` and `checkpoints/`

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

REPO_DIR = Path("/content/Semantic-Correspondence")
BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/AML_results")
RUNS_DIR = BASE_DIR / "runs"
CKPT_DIR = BASE_DIR / "checkpoints"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
for link_name, target in (("runs", RUNS_DIR), ("checkpoints", CKPT_DIR)):
    p = REPO_DIR / link_name
    if p.is_symlink() or p.is_file():
        p.unlink()
    elif p.is_dir():
        shutil.rmtree(p)
    os.symlink(str(target), str(p))
    print(f"Linked {p} -> {target}")


### SPair-71k

In [ ]:
import os
import sys
import tarfile
import urllib.request
from pathlib import Path

REPO_DIR = Path("/content/Semantic-Correspondence")
os.chdir(REPO_DIR)

SPAIR_URL = "https://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz"
DATA_DIR = REPO_DIR / "data"
SPAIR_ROOT = DATA_DIR / "SPair-71k"
TAR_PATH = DATA_DIR / "SPair-71k.tar.gz"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not SPAIR_ROOT.is_dir():
    if not TAR_PATH.is_file():
        print("Downloading:", SPAIR_URL)
        urllib.request.urlretrieve(SPAIR_URL, TAR_PATH)
    print("Extracting to:", DATA_DIR)
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        if sys.version_info >= (3, 12):
            tar.extractall(path=DATA_DIR, filter="data")
        else:
            tar.extractall(path=DATA_DIR)

print("SPAIR_ROOT =", SPAIR_ROOT, "| exists =", SPAIR_ROOT.is_dir())


### Install package

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/Semantic-Correspondence")
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"], check=True)
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
print("OK:", REPO_DIR)


### Pretrained weights

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/Semantic-Correspondence")
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "scripts/download_pretrained_weights.py"], check=True)


### `config.yaml`

H100-oriented defaults: `precision: auto` → bf16 on CUDA; DINO FT 32 / LoRA 48 / SAM 8; `num_workers: 1`; `resume_save_interval: 25`.

In [ ]:
import os
from pathlib import Path

import yaml

REPO_DIR = Path("/content/Semantic-Correspondence")
os.chdir(REPO_DIR)

START_FROM_SCRATCH = False
DRIVE_AML_BASE = Path("/content/drive/MyDrive/Colab Notebooks/AML_results")
CHECKPOINT_DIR_ON_DRIVE = str(DRIVE_AML_BASE / "checkpoints")

FT_BATCH_SIZE_BY_BACKBONE = {
    "dinov2_vitb14": 32,
    "dinov3_vitb16": 32,
    "sam_vit_b": 8,
}
LORA_BATCH_SIZE_BY_BACKBONE = {
    "dinov2_vitb14": 48,
    "dinov3_vitb16": 48,
    "sam_vit_b": 8,
}

cfg = {
    "dataset": {"backend": "sd4match", "metrics_backend": "sd4match"},
    "paths": {
        "spair_root": str(REPO_DIR / "data" / "SPair-71k"),
        "checkpoint_dir": CHECKPOINT_DIR_ON_DRIVE,
    },
    "runtime": {
        "device": "cuda",
        "precision": "auto",
        "num_workers": 1,
        "preprocess": "FIXED_RESIZE",
        "image_size_by_backbone": {
            "dinov2_vitb14": [518, 518],
            "dinov3_vitb16": [512, 512],
            "sam_vit_b": [512, 512],
        },
        "limit_pairs": 0,
        "eval_split": "test",
        "alphas": [0.05, 0.1, 0.2],
        "wsa_window": 5,
        "wsa_temperature": 1.0,
        "log_batch_interval": 25,
        "resume_save_interval": 25,
    },
    "finetune": {
        "last_blocks": [1, 2, 4],
        "epochs": 50,
        "patience": 7,
        "batch_size": 32,
        "batch_size_by_backbone": FT_BATCH_SIZE_BY_BACKBONE,
        "lr": 5e-5,
        "weight_decay": 0.01,
        "dinov2_weights": str(REPO_DIR / "checkpoints" / "dinov2_vitb14_pretrain.pth"),
        "dinov3_weights": str(
            REPO_DIR / "checkpoints" / "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
        ),
        "sam_checkpoint": str(REPO_DIR / "checkpoints" / "sam_vit_b_01ec64.pth"),
    },
    "lora": {
        "rank": 8,
        "alpha": 16.0,
        "lr": 1e-3,
        "last_blocks": 2,
        "epochs": 50,
        "patience": 7,
        "batch_size": 48,
        "batch_size_by_backbone": LORA_BATCH_SIZE_BY_BACKBONE,
    },
    "workflow_toggles": {
        "run_verify_dataset": True,
        "train_finetune": [True, True, True],
        "train_lora": [True, True, True],
        "run_eval_baseline": [True, True, True],
        "run_eval_baseline_wsa": [True, True, True],
        "run_eval_finetuned_checkpoint": [True, True, True],
        "run_eval_lora_checkpoint": [True, True, True],
        "run_eval_finetuned_wsa": [True, True, True],
        "run_eval_lora_wsa": [True, True, True],
        "run_export_metrics_tables": True,
        "run_pytest": False,
        "pipeline_resume": not START_FROM_SCRATCH,
    },
}

cfg_path = REPO_DIR / "config.yaml"
with open(cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)

print("Written:", cfg_path)
print("checkpoint_dir:", cfg["paths"]["checkpoint_dir"])
print("resume_save_interval:", cfg["runtime"]["resume_save_interval"])
print("finetune batch map:", cfg["finetune"]["batch_size_by_backbone"])
print("lora batch map:", cfg["lora"]["batch_size_by_backbone"])


### Run pipeline

In [ ]:
import json
import os
import subprocess
import sys
import threading
from collections import deque
from pathlib import Path

from IPython.display import clear_output

REPO_DIR = Path("/content/Semantic-Correspondence")
os.chdir(REPO_DIR)

env = dict(os.environ)
if START_FROM_SCRATCH:
    env["SEMANTIC_CORRESPONDENCE_PIPELINE_RESET"] = "1"
else:
    env.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_RESET", None)
env.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_LOG_FILE_ONLY", None)
env["PYTHONUNBUFFERED"] = "1"
_pp = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = f"{REPO_DIR}{os.pathsep}{_pp}" if _pp else str(REPO_DIR)

STAGE_EVENTS_PATH = REPO_DIR / "runs" / "logs" / "stage_events.jsonl"
TAIL_LINES = 40
REFRESH_SECONDS = 3.0
cmd = [sys.executable, "-u", "scripts/run_pipeline.py", "--config", "config.yaml"]

_output_lines: deque = deque(maxlen=TAIL_LINES)
_proc_done = threading.Event()
_return_code: list = []


def _stream_subprocess() -> None:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
        cwd=str(REPO_DIR),
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        _output_lines.append(line.rstrip())
    _return_code.append(proc.wait())
    _proc_done.set()


threading.Thread(target=_stream_subprocess, daemon=True).start()


def _read_stage_events() -> list:
    if not STAGE_EVENTS_PATH.is_file():
        return []
    events = []
    with open(STAGE_EVENTS_PATH, "r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if not raw:
                continue
            try:
                events.append(json.loads(raw))
            except json.JSONDecodeError:
                pass
    return events


def _render(events: list) -> None:
    done = [e["stage_id"] for e in events if e.get("action") == "done"]
    skipped = [e["stage_id"] for e in events if e.get("action") == "skip"]
    failed = [e["stage_id"] for e in events if e.get("action") == "fail"]
    latest_start = next((e for e in reversed(events) if e.get("action") == "start"), None)
    current = latest_start["stage_id"] if latest_start else "(waiting...)"
    rc = _return_code[0] if _return_code else None
    if not _proc_done.is_set():
        status = "RUNNING"
    elif rc == 0:
        status = "COMPLETED"
    else:
        status = f"FAILED (exit={rc})"
    sep = "=" * 64
    print(sep)
    print(f"  Pipeline: {status}")
    print(f"  Current stage: {current}")
    print(f"  Completed: {len(done):3d}  Skipped: {len(skipped):3d}  Failed: {len(failed):3d}")
    if done:
        tail = done[-6:]
        suffix = "..." if len(done) > 6 else ""
        print(f"  Last done: {', '.join(tail)}{suffix}")
    if failed:
        print(f"  FAILED: {', '.join(failed)}")
    print(sep)
    print(f"--- last {TAIL_LINES} log lines ---")
    for line in _output_lines:
        print(line)


while not _proc_done.is_set():
    clear_output(wait=True)
    _render(_read_stage_events())
    _proc_done.wait(timeout=REFRESH_SECONDS)

clear_output(wait=True)
_render(_read_stage_events())

rc = _return_code[0] if _return_code else -1
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)
print("\nPipeline completed successfully.")


### Stage summary

In [ ]:
import json
from pathlib import Path

import pandas as pd

REPO_DIR = Path("/content/Semantic-Correspondence")
p = REPO_DIR / "runs" / "logs" / "stage_events.jsonl"
if not p.is_file():
    print("No file:", p)
else:
    events = []
    with open(p, "r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if raw:
                try:
                    events.append(json.loads(raw))
                except json.JSONDecodeError:
                    pass
    done = [e["stage_id"] for e in events if e.get("action") == "done"]
    skipped = [e["stage_id"] for e in events if e.get("action") == "skip"]
    failed = [e["stage_id"] for e in events if e.get("action") == "fail"]
    print(f"events={len(events)} done={len(done)} skipped={len(skipped)} failed={len(failed)}")
    if failed:
        print("Failed:", failed)
    display(pd.DataFrame(events).tail(20))


### Training curves

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

REPO_DIR = Path("/content/Semantic-Correspondence")
CKPT_DIR = REPO_DIR / "checkpoints"
files = sorted(CKPT_DIR.glob("*_history.jsonl"))
if not files:
    print("No history in", CKPT_DIR)
else:
    histories = {}
    for hf in files:
        name = hf.stem.replace("_history", "")
        recs = []
        with open(hf, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    recs.append(json.loads(line))
        if recs:
            histories[name] = recs
    if histories:
        n = len(histories)
        fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), squeeze=False)
        for col, (name, records) in enumerate(sorted(histories.items())):
            ax = axes[0, col]
            ep = [r["epoch"] for r in records]
            ax.plot(ep, [r["train_loss"] for r in records], marker=".", label="train_loss")
            ax.plot(ep, [r["val_loss"] for r in records], marker=".", label="val_loss")
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Loss")
            ax.set_title(name, fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
        fig.suptitle("Training curves", fontsize=13)
        fig.tight_layout()
        plt.show()
    else:
        print("Empty history files.")


## Results

### Load exports

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

REPO_DIR = Path("/content/Semantic-Correspondence")
os.chdir(REPO_DIR)
EXPORTS = REPO_DIR / "runs" / "pipeline_exports"


def _load_json(name):
    p = EXPORTS / name
    if p.is_file():
        with open(p, encoding="utf-8") as f:
            return json.load(f)
    print("Not found:", p)
    return None


pck_results = _load_json("pck_results.json")
per_category = _load_json("pck_results_per_category.json")
by_difficulty = _load_json("pck_results_by_difficulty_flag.json")
print("Loaded", len(pck_results or []), "eval runs.")


### Aggregate PCK

In [ ]:
if pck_results:
    rows = []
    for r in pck_results:
        row = {"name": r["name"]}
        row.update(r.get("metrics", {}))
        rows.append(row)
    df_pck = pd.DataFrame(rows)
    display(
        df_pck.style.format({c: "{:.4f}" for c in df_pck.columns if c.startswith("pck@")}).set_caption(
            "PCK"
        )
    )
else:
    print("No results.")


### Fine-tune depth

In [ ]:
import re

import matplotlib.pyplot as plt

if pck_results:
    ft_rows = []
    for r in pck_results:
        m = re.match(r"(.+)_ft_lb(\d+)$", r["name"])
        if m:
            row = {"backbone": m.group(1), "last_blocks": int(m.group(2))}
            row.update(r.get("metrics", {}))
            ft_rows.append(row)
    if ft_rows:
        df_ft = pd.DataFrame(ft_rows).sort_values(["backbone", "last_blocks"])
        display(df_ft)
        fig, axes = plt.subplots(
            1, len(df_ft["backbone"].unique()), figsize=(5 * len(df_ft["backbone"].unique()), 4), squeeze=False
        )
        for col_idx, (bb, grp) in enumerate(df_ft.groupby("backbone")):
            ax = axes[0, col_idx]
            for mc in [c for c in grp.columns if c.startswith("pck@")]:
                ax.plot(grp["last_blocks"], grp[mc], marker="o", label=mc)
            ax.set_xlabel("Unfrozen blocks")
            ax.set_ylabel("PCK")
            ax.set_title(bb)
            ax.legend(fontsize=8)
            ax.set_xticks(sorted(grp["last_blocks"].unique()))
        fig.suptitle("PCK vs fine-tuned blocks", fontsize=13)
        fig.tight_layout()
        plt.show()
    else:
        print("No *_ft_lb<N> rows.")
else:
    print("No pck_results.")


### Per-category PCK

In [ ]:
import numpy as np

if per_category:
    rows = []
    for entry in per_category:
        for cat, alphas in entry.get("categories", {}).items():
            row = {"run": entry["name"], "category": cat}
            row.update(alphas)
            rows.append(row)
    if rows:
        df_cat = pd.DataFrame(rows)
        pck_col = [c for c in df_cat.columns if c.startswith("pck@")]
        if pck_col:
            pivot = df_cat.pivot_table(index="category", columns="run", values=pck_col[0], aggfunc="first")
            fig, ax = plt.subplots(figsize=(max(12, len(pivot.columns) * 1.5), max(6, len(pivot) * 0.4)))
            im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=8)
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels(pivot.index, fontsize=8)
            for i in range(pivot.shape[0]):
                for j in range(pivot.shape[1]):
                    v = pivot.values[i, j]
                    if not np.isnan(v):
                        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
            fig.colorbar(im, ax=ax, label=pck_col[0])
            ax.set_title(f"Per-category {pck_col[0]}")
            fig.tight_layout()
            plt.show()
    else:
        print("No per-category rows.")
else:
    print("No per_category export.")


### Per-difficulty

In [ ]:
if by_difficulty:
    rows = []
    for entry in by_difficulty:
        run_name = entry["name"]
        for flag, buckets in entry.get("data", {}).items():
            for bucket, info in buckets.items():
                summary = info.get("summary", {}).get("image", {})
                for metric_key, val_dict in summary.items():
                    rows.append(
                        {
                            "run": run_name,
                            "flag": flag,
                            "value": int(bucket),
                            "metric": metric_key.replace("custom_", ""),
                            "pck": val_dict.get("all", float("nan")),
                        }
                    )
    if rows:
        import pandas as pd

        df_diff = pd.DataFrame(rows)
        display(df_diff.pivot_table(index=["run", "metric"], columns=["flag", "value"], values="pck").round(4))
    else:
        print("No difficulty rows.")
else:
    print("No difficulty export.")


### Qualitative (DINOv2 baseline)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from data.dataset import PreprocessMode, SPair71kPairDataset
from evaluation.visualize import visualize_correspondences
from models.common import (
    BackboneName,
    DenseExtractorConfig,
    DenseFeatureExtractor,
    predict_correspondences_cosine_argmax,
)

REPO_DIR = Path("/content/Semantic-Correspondence")
spair_root = str(REPO_DIR / "data" / "SPair-71k")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ds = SPair71kPairDataset(
    spair_root=spair_root,
    split="test",
    preprocess=PreprocessMode.FIXED_RESIZE,
    output_size_hw=(784, 784),
    normalize=True,
)
cfg_ext = DenseExtractorConfig(
    name=BackboneName.DINOV2_VIT_B14,
    dinov2_weights_path=str(REPO_DIR / "checkpoints" / "dinov2_vitb14_pretrain.pth"),
)
extractor = DenseFeatureExtractor(cfg_ext).to(device).eval()

NUM_EXAMPLES = 4
indices = list(range(0, min(len(ds), NUM_EXAMPLES * 50), max(1, len(ds) // NUM_EXAMPLES)))[:NUM_EXAMPLES]

for idx in indices:
    sample = ds[idx]
    src = sample["src_img"].unsqueeze(0).to(device)
    tgt = sample["tgt_img"].unsqueeze(0).to(device)
    src_kps = sample["src_kps"].to(device)
    tgt_kps = sample["tgt_kps"].to(device)
    pck_thr = float(sample["pck_threshold_bbox"])
    with torch.no_grad():
        feat_src, _ = extractor(src)
        feat_tgt, _ = extractor(tgt)
        out = predict_correspondences_cosine_argmax(
            feat_src,
            feat_tgt,
            src_kps,
            img_hw=(784, 784),
            img_hw_src=(784, 784),
            img_hw_tgt=(784, 784),
        )
        pred_tgt = out["pred_tgt_xy"]
    fig = visualize_correspondences(
        sample["src_img"],
        sample["tgt_img"],
        src_kps,
        pred_tgt,
        tgt_kps,
        pck_threshold=pck_thr,
        alpha=0.1,
        title=f"Pair {idx} (category: {sample.get('category', '?')})",
    )
    plt.show()
    plt.close(fig)
